# Counseling Progress Predictive

## Problem Framing

**Business question:** Which residents in active counseling are most likely to show near-term progress?

**Operational decision supported:** Help social workers spot which support patterns are most associated with near-term counseling progress.

**Primary users:** social workers, case managers

**Target summary:** Current predictive label: `label_counseling_progress_next_90d`, based on future sessions with strong progress notes, positive end states, and low concern rates.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='counseling_progress',
    dataset_name='resident_monthly_features',
    predictive_impl='counseling_progress',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,50,2023-02-01,4,Active,Neglected,F,High,1,11.98,2,...,True,True,True,False,False,0,True,False,False,False
1,50,2023-03-01,4,Active,Neglected,F,High,2,12.06,2,...,True,True,True,False,False,1,True,False,False,False
2,13,2023-04-01,2,Closed,Surrendered,F,Medium,2,8.89,1,...,True,True,True,False,False,1,True,False,False,False
3,29,2023-04-01,8,Transferred,Abandoned,F,Medium,1,12.17,1,...,True,True,True,False,True,1,True,False,False,False
4,23,2023-04-01,5,Closed,Foundling,F,High,2,9.23,1,...,True,True,True,False,True,1,True,False,False,False
5,50,2023-04-01,4,Active,Neglected,F,High,3,12.15,2,...,True,True,True,False,False,1,True,False,False,False
6,13,2023-05-01,2,Closed,Surrendered,F,Medium,3,8.97,1,...,True,True,True,False,False,1,False,False,False,False
7,36,2023-05-01,1,Active,Surrendered,F,High,2,12.23,2,...,True,True,True,False,False,1,True,True,False,False
8,11,2023-05-01,1,Closed,Surrendered,F,Low,1,17.46,1,...,True,True,True,False,False,0,True,False,False,False
9,45,2023-05-01,3,Transferred,Abandoned,F,Medium,3,15.24,2,...,True,True,True,False,False,0,False,False,False,False


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_counseling_progress_next_90d`, based on future sessions with strong progress notes, positive end states, and low concern rates.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('counseling_progress')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'random_forest_classifier',
 'train_rows': 655,
 'test_rows': 485,
 'target_col': 'label_counseling_progress_next_90d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 485.0,
 'positive_count': 221.0,
 'positive_rate': 0.4556701030927835,
 'accuracy': 0.5154639175257731,
 'balanced_accuracy': 0.5110722610722611,
 'precision': 0.46788990825688076,
 'recall': 0.46153846153846156,
 'f1': 0.4646924829157175,
 'roc_auc': 0.5184252022487317,
 'average_precision': 0.5024179125798504}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,counseling_progress,random_forest_classifier,485.0,221.0,0.45567,0.515464,0.511072,0.467890,0.461538,0.464692,0.518425,0.502418,NaN,NaN,NaN
1,counseling_progress,logistic_regression,485.0,221.0,0.45567,0.529897,0.509958,0.473684,0.285068,0.355932,0.495578,0.476652,NaN,NaN,NaN
2,counseling_progress,dummy_classifier,485.0,221.0,0.45567,0.455670,0.500000,0.455670,1.000000,0.626062,0.500000,0.455670,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,counseling_progress,random_forest_classifier,3.0,1.581139,131.0,0.0,70.2,0.447214,0.535878,0.003414,...,0.059854,0.758101,0.071636,5,NaN,NaN,NaN,NaN,NaN,NaN
1,counseling_progress,logistic_regression,3.0,1.581139,131.0,0.0,70.2,0.447214,0.535878,0.003414,...,0.028308,0.670819,0.036670,5,NaN,NaN,NaN,NaN,NaN,NaN
2,counseling_progress,dummy_classifier,3.0,1.581139,131.0,0.0,70.2,0.447214,0.535878,0.003414,...,0.000000,0.535878,0.003414,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to social workers, case managers?


## Deployment Notes

Recommended shared widgets:

* `insight_summary_card`
* `ranked_table_widget`
* `explanation_chart_card`

Deployment checklist:

* Use a progress card on resident profiles and a ranked view during counseling planning.
* Keep the explanation summary close to the score so staff can see which recent patterns matter most.

Standard endpoint pattern:

* `POST /ml/predict/counseling_progress`
* `POST /ml/score-batch/counseling_progress`
